## 10.8 — קריאת קוד pandas: קוד שכתב אייג'נט ובאגים אופייניים

זה הסעיף המרכזי של "אוריינות ולא הפקה": בפועל, כשנותנים לסוכן בינה מלאכותית קובץ CSV ומבקשים ניתוח, הוא כמעט תמיד יכתוב pandas. השאלה החשובה היא לא "איך כותבים את זה" אלא **"איך קוראים את זה ובודקים שהוא נכון"**.

```{admonition} 🎥 כאן נכנס סרטון השבוע
:class: seealso

**"לקרוא קוד pandas שכתב אייג'נט"** (כ-6–10 דקות, הקלטת מסך של המחברת עם קול).

<!-- TODO (למפיק/ה): להחליף תא זה בהטמעת הסרטון בפועל, למשל:
<iframe width="100%" height="400" src="VIDEO_URL_HERE" title="שבוע 10 — לקרוא קוד pandas שכתב אייג'נט" frameborder="0" allowfullscreen></iframe>
-->
```

### דוגמה: סקירת קוד

נניח שביקשנו מסוכן AI: *"טען את launch_log.csv, ומצא את הזווית עם הטווח הממוצע הגבוה ביותר, עבור מדידות בהן v0 היה גדול מ-19."* הוא החזיר את הקוד הבא. לפני שמריצים אותו — קראו אותו שורה-שורה, ונסו למצוא בעיות.

In [ ]:
import pandas as pd

df = pd.read_csv("launch_log.csv")

fast = df[df["v0_measured"] > 19]
grouped = fast.groupby("angle_deg")["range_measured"]
best_angle = grouped.max()

print("הזווית הכי טובה:", best_angle)

### נסו בעצמכם

לפני שתמשיכו לפתרון — מה, לדעתכם, `best_angle` בפועל מכיל? האם זו זווית בודדת (מספר), או משהו אחר?

In [ ]:
# תחזית במילים (אין קוד להריץ כאן)

`````{admonition} פתרון
:class: dropdown, tip
`best_angle` הוא בעצם `Series` שלם (הטווח **המקסימלי** בכל קבוצת זווית), לא זווית בודדת — למרות שהמשתנה נקרא `best_angle` וההדפסה נראית כאילו היא "התשובה". יש כאן שתי בעיות בפועל: (1) `grouped.max()` מחזיר את הטווח המקסימלי בכל קבוצה, לא את **הממוצע** שהתבקש; (2) גם אחרי תיקון ל-`.mean()`, עדיין מתקבל `Series` עם 5 ערכים (אחד לכל זווית) — לא "הזווית הטובה ביותר" עצמה. חסר שלב אחרון: `.idxmax()` על התוצאה, כדי לקבל את **תווית הזווית** בעלת הערך הגבוה ביותר.
`````

### הבאגים המלאים ותיקונם

בואו נעבור שיטתית: מה הקוד *התכוון* לעשות, ומה הוא *עושה בפועל*.

In [ ]:
# הקוד המקורי, לצורך השוואה:
fast = df[df["v0_measured"] > 19]
grouped = fast.groupby("angle_deg")["range_measured"]
best_angle_wrong = grouped.max()
print(type(best_angle_wrong))
print(best_angle_wrong)

`````{admonition} פתרון
:class: dropdown, tip
`````{admonition} פתרון — שני באגים
:class: dropdown, tip
**באג 1: `max()` במקום `mean()`.** הבקשה הייתה "הטווח הממוצע הגבוה ביותר", אבל הקוד קרא ל-`grouped.max()` — הטווח **המקסימלי שנמדד אי פעם** בכל קבוצה, לא הממוצע. שתי סטטיסטיקות שונות לגמרי; קל להחליף ביניהן כי שתיהן "מקסימום משהו".

**באג 2: התוצאה היא Series שלם, לא זווית בודדת.** גם אחרי תיקון ל-`.mean()`, `grouped.mean()` מחזיר `Series` עם ערך לכל זווית (5 שורות) — לא "הזווית הטובה ביותר". כדי לקבל את תווית הזווית בעלת הערך הגבוה ביותר, צריך `.idxmax()`:

```python
fast = df[df["v0_measured"] > 19]
mean_range_by_angle = fast.groupby("angle_deg")["range_measured"].mean()
best_angle = mean_range_by_angle.idxmax()
best_value = mean_range_by_angle.max()

print(f"הזווית עם הטווח הממוצע הגבוה ביותר: {best_angle}° (טווח ממוצע {best_value:.1f}m)")
```
`````
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "קוד שכתב סוכן AI קורא ל-<code>df.groupby(\"angle_deg\")[\"range_measured\"].max()</code> כשהמשימה הייתה למצוא את הממוצע הגבוה ביותר. מה הבעיה?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "אין בעיה, max ו-mean שקולים כאן", "correct": False, "feedback": "לא — הן סטטיסטיקות שונות לגמרי."},
            {"answer": "max() מחזיר את הערך הבודד הגבוה ביותר שנמדד בכל קבוצה, לא את הממוצע של הקבוצה", "correct": True, "feedback": "נכון."},
            {"answer": "groupby לא תומך בקריאה ל-max()", "correct": False, "feedback": "לא נכון — זו קריאה תקינה."}
        ]
    },
    {
        "question": "יש לכם Series עם ממוצע הטווח לכל זווית. איך מוצאים את תווית הזווית בעלת הממוצע הגבוה ביותר (לא רק את הערך)?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "עם .max()", "correct": False, "feedback": "זה נותן רק את הערך המקסימלי, לא את התווית."},
            {"answer": "עם .idxmax()", "correct": True, "feedback": "נכון."},
            {"answer": "עם .sort_values() בלבד", "correct": False, "feedback": "מיון לא נותן ישירות את התווית המקסימלית בלי שלב נוסף."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

הקוד הבא (שוב, "כתיבת אייג'נט") נועד "לנקות" ערכי טווח שליליים (בלתי אפשריים פיזיקלית) — להחליף אותם ב-`NaN` בטבלה `df` עצמה. יש בו באג — מצאו ותקנו. (רמז: הריצו את הקוד ואז בדקו אם `df` באמת השתנה.)

In [ ]:
import numpy as np
df_check = df.copy()
df_check.loc[0, "range_measured"] = -5.0   # מדמים כאן שגיאת מדידה, לצורך התרגיל

df_check[df_check["range_measured"] < 0]["range_measured"] = np.nan   # <- שימו לב כאן
print((df_check["range_measured"] < 0).sum(), "ערכים שליליים נשארו")

`````{admonition} פתרון
:class: dropdown, tip
`````{admonition} פתרון
:class: dropdown, tip
`df_check[df_check["range_measured"] < 0]` יוצר **עותק זמני** של השורות המסוננות — לא "חלון" לתוך `df_check` המקורי. ההשמה `["range_measured"] = np.nan` שאחריו כותבת לתוך העותק הזמני הזה, שנזרק מיד — `df_check` המקורי לא משתנה בכלל. זה בדיוק סוג הבאג "מוטציה של מערך הקלט": הקוד רץ (אולי עם אזהרה בלבד), אבל בלי לעשות את מה שהתכוונו.

הפתרון: לבצע את הבחירה וההשמה **בפעולה אחת**, עם `.loc`:

```python
df_check.loc[df_check["range_measured"] < 0, "range_measured"] = np.nan
print((df_check["range_measured"] < 0).sum(), "ערכים שליליים נשארו")   # 0
```
`````
`````